# Google ADK Agents Workbook

All ADK work for this project lives here.

- **Part 1 – First workflow:** coordinator → Data Analyst → SQL tool.
- **Part 2 – Multi-agent decision system:** Knowledge, Research, Critic and Decision agents sharing evidence-backed state.

The sample tenant and records are fictional and local-only. Live runs need `GOOGLE_API_KEY`, `DATABASE_URL` and `QDRANT_URL` in the root `.env`. Free-tier keys are throttled to ~5 requests/minute, so Part 2 paces model calls (about 13s apart) and takes several minutes.

## 1. Update Python Project for ADK Workflow

The application code lives in `src/ai_template_python`: ADK topology in `agents/`, the read-only query in `tools/sql/`, shared investigation contracts in `contracts.py`, and versioned state in `state.py`. The project dependency is locked in `pyproject.toml` and `uv.lock`.

In [1]:
from pathlib import Path
import tomllib

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").is_file():
    ROOT = next(parent for parent in ROOT.parents if (parent / "pyproject.toml").is_file())

PROJECT = tomllib.loads((ROOT / "pyproject.toml").read_text(encoding="utf-8"))
assert "google-adk>=2.11.0,<3.0.0" in PROJECT["project"]["dependencies"]
for relative_path in (
    "src/ai_template_python/agents/first_investigation.py",
    "src/ai_template_python/tools/sql/spend_analysis.py",
    "src/ai_template_python/contracts.py",
    "src/ai_template_python/state.py",
):
    assert (ROOT / relative_path).is_file(), relative_path

print(f"Project ready: {PROJECT['project']['name']}")
print("ADK dependency and workflow modules are present.")

Project ready: ai-template-python
ADK dependency and workflow modules are present.


## 2. Install and Configure Google ADK in Notebook

Install the locked project and notebook tools with `uv sync --group notebook`, then select that environment as the Jupyter kernel. If using a non-project kernel, install the workspace package with `%pip install -e .` and restart the kernel. Set `GOOGLE_API_KEY` in the ignored root `.env`; never put the key in a notebook cell.

In [2]:
import os
from importlib.metadata import version

from dotenv import load_dotenv

load_dotenv(ROOT / ".env", override=False)

from google.adk import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

MODEL = os.getenv("GEMINI_MODEL", "gemini-flash-latest")
TENANT_ID = "00000000-0000-4000-8000-000000000001"  # trusted local synthetic tenant
APP_NAME = "enterprise_decision_intelligence"
USER_ID = "local-notebook-user"

print(f"Google ADK: {version('google-adk')}")
print(f"Model configured: {MODEL}")
print(f"GOOGLE_API_KEY present: {bool(os.getenv('GOOGLE_API_KEY'))}")
print(f"DATABASE_URL present: {bool(os.getenv('DATABASE_URL'))}")

Google ADK: 2.11.0
Model configured: gemini-flash-lite-latest
GOOGLE_API_KEY present: True
DATABASE_URL present: True


## 3. Define Shared Models and Agent State

The Python package keeps contracts framework-independent and uses versioned JSON-serializable investigation state. This cell shows the same request and state shape that the SQL analyst handles.

In [3]:
from datetime import date

from ai_template_python.contracts import SpendAnalysisRequest
from ai_template_python.state import new_investigation_state

request = SpendAnalysisRequest(
    start_date=date(2024, 1, 1),
    end_date=date(2025, 12, 31),
    supplier_id="S102",
    group_by="month",
)
investigation_state = new_investigation_state(
    investigation_id="notebook-s102-spend",
    tenant_id=TENANT_ID,
    requester_id=USER_ID,
    question="Explain monthly supplier spend and the largest month-over-month change.",
)
print(request)
print({key: investigation_state[key] for key in ("schema_version", "status", "tenant_id", "question")})

SpendAnalysisRequest(start_date=datetime.date(2024, 1, 1), end_date=datetime.date(2025, 12, 31), supplier_id='S102', category_id=None, group_by='month')
{'schema_version': 1, 'status': 'queued', 'tenant_id': '00000000-0000-4000-8000-000000000001', 'question': 'Explain monthly supplier spend and the largest month-over-month change.'}


## 4. Implement Root Agent

The root coordinator delegates analytical work and returns a concise finding grounded in the specialist's output. Its instructions also prevent purchase-order spend from being misreported as paid invoice spend.

In [4]:
from ai_template_python.agents.first_investigation import root_agent

assert root_agent.name == "root_agent"
print(root_agent.name, root_agent.description)

root_agent Coordinates procurement investigations and presents evidence-based findings.


## 5. Implement Data Analyst Agent

The Data Analyst is a single-turn ADK sub-agent. It is responsible for selecting the safe analysis tool, interpreting returned figures, and clearly marking synthetic data.

In [5]:
from ai_template_python.agents.first_investigation import data_analyst_agent

assert data_analyst_agent.mode == "single_turn"
assert data_analyst_agent in root_agent.sub_agents
print(data_analyst_agent.name, data_analyst_agent.description)

data_analyst Analyzes tenant-scoped supplier purchase-order spend using SQL.


## 6. Create and Register SQL Tool

The analyst receives a Python function tool, not arbitrary SQL. SQL text is fixed in the module; supplier and date values are bound parameters, and tenant scope is injected from trusted ADK session state.

In [6]:
import inspect

from ai_template_python.tools.sql.spend_analysis import analyze_supplier_monthly_spend

tool_parameters = inspect.signature(analyze_supplier_monthly_spend).parameters
assert "supplier_id" in tool_parameters
assert "start_date" in tool_parameters and "end_date" in tool_parameters
assert "tenant_id" not in tool_parameters
assert "sql" not in tool_parameters
assert analyze_supplier_monthly_spend in data_analyst_agent.tools
print("Registered safe SQL tool:", analyze_supplier_monthly_spend.__name__)
print("LLM inputs:", [name for name in tool_parameters if name != "tool_context"])

Registered safe SQL tool: analyze_supplier_monthly_spend
LLM inputs: ['supplier_id', 'start_date', 'end_date']


## 7. Wire Root Agent to Data Analyst and SQL Tool

ADK registers the Data Analyst as a sub-agent and automatically provides delegation to the root. The Runner session receives the authorized local tenant ID as session state; it is not part of the user prompt or tool arguments.

In [7]:
session_service = InMemorySessionService()
runner = Runner(
    agent=root_agent,
    app_name=APP_NAME,
    session_service=session_service,
)

print("Runner configured for:", root_agent.name)
print("Delegates:", [agent.name for agent in root_agent.sub_agents])

App "enterprise_decision_intelligence" can transfer between agents but has no context_cache_config. Every transfer swaps the system instruction and the tool set, so the request prefix changes and the whole prompt is re-sent uncached after each transfer. Set context_cache_config on the app to give each agent its own cache.


Runner configured for: root_agent
Delegates: ['data_analyst']


## 8. Run One End-to-End Investigation

Ask the analyst to summarize S102's 2024–2025 monthly GBP purchase-order spend and identify its largest absolute month-over-month change. The returned events include specialist/root handoffs and tool activity; live execution is skipped when either credential or database configuration is missing.

In [8]:
async def run_investigation(question: str) -> dict[str, object]:
    """Run one ADK investigation using a trusted local tenant session state."""
    if not os.getenv("GOOGLE_API_KEY"):
        raise RuntimeError("Set GOOGLE_API_KEY in the ignored root .env before a live model run.")
    if not os.getenv("DATABASE_URL"):
        raise RuntimeError("Set DATABASE_URL in the environment before a database investigation.")

    session = await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        state={"tenant_id": TENANT_ID},
    )
    message = types.Content(role="user", parts=[types.Part(text=question)])
    trace: dict[str, object] = {"agents": [], "tools": [], "errors": []}
    event_authors: list[str] = trace["agents"]  # type: ignore[assignment]
    tool_authors: list[str] = trace["tools"]  # type: ignore[assignment]

    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=session.id,
        new_message=message,
    ):
        if event.author not in event_authors:
            event_authors.append(event.author)
        if event.author not in {"root_agent", "data_analyst"} and event.author not in tool_authors:
            tool_authors.append(event.author)
        if event.content and event.content.parts:
            text = "".join(part.text or "" for part in event.content.parts)
            if text.strip():
                print(f"[{event.author}] {text.strip()}")

    return trace


async def run_with_free_tier_retry(question: str, attempts: int = 4) -> dict[str, object]:
    """Retry on Gemini free-tier 429/503 errors, waiting out the per-minute quota window."""
    import asyncio
    import re

    for attempt in range(1, attempts + 1):
        try:
            return await run_investigation(question)
        except Exception as error:  # noqa: BLE001
            message = str(error)
            if attempt == attempts or not any(code in message for code in ("429", "503")):
                raise
            match = re.search(r"retry in ([\d.]+)s", message)
            delay = float(match.group(1)) + 5 if match else 65
            print(f"Attempt {attempt} hit a free-tier limit; retrying in {delay:.0f}s")
            await asyncio.sleep(delay)

In [9]:
trace = await run_with_free_tier_retry(
    "summarize S102's 2024–2025 monthly GBP purchase-order spend and identify its largest absolute month-over-month change"
)
trace

/home/rajesh/projects/Portfolio/enterprise-decision-intelligence/.venv/lib/python3.13/site-packages/google/adk/models/llm_request.py:306: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()


[data_analyst] Here is the monthly GBP purchase-order spend for supplier **S102** for the years 2024 and 2025. 

*(Note: The returned data is synthetic).*

### Monthly Spend Breakdown
* **2024-01-01:** £2,400.00
* **2024-02-01:** £2,204.40
* **2024-03-01:** £1,204.80
* **2024-04-01:** £2,313.80
* **2024-05-01:** £1,990.80
* **2024-06-01:** £1,818.00
* **2024-07-01:** £1,087.90
* **2024-08-01:** £2,104.05
* **2024-09-01:** £1,371.60
* **2024-10-01:** £1,908.75
* **2024-11-01:** £2,371.50
* **2024-12-01:** £1,814.05
* **2025-01-01:** £1,254.40
* **2025-02-01:** £1,385.10
* **2025-03-01:** £1,516.30
* **2025-04-01:** £1,570.75
* **2025-05-01:** £2,089.80
* **2025-06-01:** £1,576.85
* **2025-07-01:** £1,250.76
* **2025-08-01:** £1,671.04
* **2025-09-01:** £1,524.90
* **2025-10-01:** £2,157.12
* **2025-11-01:** £2,881.92
* **2025-12-01:** £2,525.88

---

### Largest Month-over-Month Change
* **Underlying Months:** March 2024 (£1,204.80) to April 2024 (£2,313.80)
* **Absolute Amount Change:*

[root_agent] ### Summary of Supplier S102 Purchase-Order Spend (2024–2025)

Based on the analysis of tenant-scoped data, the monthly purchase-order spend in **GBP** for supplier **S102** across 2024 and 2025 is detailed below. 

*Please note: These figures represent purchase-order values, not necessarily paid invoices, and are derived from synthetic data.*

#### Monthly Spend Breakdown
* **2024:**
  * January: £2,400.00
  * February: £2,204.40
  * March: £1,204.80
  * April: £2,313.80
  * May: £1,990.80
  * June: £1,818.00
  * July: £1,087.90
  * August: £2,104.05
  * September: £1,371.60
  * October: £1,908.75
  * November: £2,371.50
  * December: £1,814.05

* **2025:**
  * January: £1,254.40
  * February: £1,385.10
  * March: £1,516.30
  * April: £1,570.75
  * May: £2,089.80
  * June: £1,576.85
  * July: £1,250.76
  * August: £1,671.04
  * September: £1,524.90
  * October: £2,157.12
  * November: £2,881.92
  * December: £2,525.88

#### Largest Absolute Month-over-Month Change
* **Per

{'agents': ['root_agent', 'data_analyst'], 'tools': [], 'errors': []}

# Part 2 – Multi-agent decision system

```text
procurement_coordinator
 ├─ data_analyst      SQL spend analysis          → analyze_supplier_monthly_spend
 ├─ knowledge_agent   contracts + policy          → get_contract_terms, search_documents (Qdrant)
 ├─ research_agent    external market context     → get_commodity_trend (World Bank)
 ├─ critic_agent      challenges the findings     → list/check_findings, record_critique
 └─ decision_agent    evidence-cited brief        → submit_decision_brief
```

**Shared state** (ADK session state) holds `tenant_id`, `evidence`, `findings`, `critique` and `decision_brief`. **Delegation** is by the coordinator through ADK sub-agents. **Evidence model:** every tool registers what it read (`E1`, `E2`, …); findings may only cite registered evidence; the brief is refused until the Critic has reviewed, and is built only from cited findings.

In [10]:
from ai_template_python.agents.decision_system import create_decision_system

team = create_decision_system(MODEL)
print(team.name, "->", [agent.name for agent in team.sub_agents])
for agent in team.sub_agents:
    print(f"{agent.name:16} tools: {[getattr(t, '__name__', str(t)) for t in agent.tools]}")

procurement_coordinator -> ['data_analyst', 'knowledge_agent', 'research_agent', 'critic_agent', 'decision_agent']
data_analyst     tools: ['analyze_supplier_monthly_spend', 'record_finding']
knowledge_agent  tools: ['get_contract_terms', 'search_documents', 'record_finding']
research_agent   tools: ['get_commodity_trend', 'record_finding']
critic_agent     tools: ['list_findings', 'check_findings', 'record_critique']
decision_agent   tools: ['list_findings', 'submit_decision_brief']


## Evidence model and shared-state tools

The tools below are exercised offline with a fake context to show the guardrails (no model call).

In [11]:
from types import SimpleNamespace

from ai_template_python.evidence import register_evidence
from ai_template_python.tools.analytics import investigation_state as state_tools

demo = SimpleNamespace(state={})
e1 = register_evidence(demo.state, source_type="database", source_id="S102", locator="demo:1",
                       citation_label="demo", excerpt="demo excerpt", is_synthetic=True)
print("registered:", e1)
print("unknown evidence :", state_tools.record_finding("data_analyst", "claim", "fact", ["E9"], "high", [], demo))
print("valid finding    :", state_tools.record_finding("data_analyst", "claim", "fact", [e1], "high", ["synthetic"], demo))
print("brief before critic:", state_tools.submit_decision_brief("h", [], "a", "r", "low", [], ["F1"], demo))

registered: E1
unknown evidence : {'status': 'error', 'message': "Unknown evidence IDs: ['E9']"}
valid finding    : {'status': 'success', 'finding_id': 'F1'}
brief before critic: {'status': 'error', 'message': 'The Critic has not reviewed the findings yet.'}


## Run the multi-agent investigation

Question: supplier S102's purchase-order spend jumped sharply in April 2024. Is there a contractual or market basis, and what should procurement do?

In [12]:
import asyncio

from ai_template_python.rate_limit import pace_model_calls  # noqa: F401  (agents already use it)

team_sessions = InMemorySessionService()
team_runner = Runner(agent=team, app_name=APP_NAME, session_service=team_sessions)

QUESTION = (
    "Supplier S102's purchase-order spend rose sharply in April 2024 compared with March 2024. "
    "Investigate between 2024-01-01 and 2025-12-31: quantify it, check contract terms and policy "
    "on price adjustments, add external commodity context, validate the evidence, and give a "
    "decision brief."
)


async def run_team(question: str):
    session = await team_sessions.create_session(
        app_name=APP_NAME, user_id=USER_ID, state={"tenant_id": TENANT_ID}
    )
    message = types.Content(role="user", parts=[types.Part(text=question)])
    async for event in team_runner.run_async(
        user_id=USER_ID, session_id=session.id, new_message=message
    ):
        for part in (event.content.parts if event.content and event.content.parts else []):
            if part.function_call:
                print(f"[{event.author}] -> {part.function_call.name}")
            elif part.text and part.text.strip():
                print(f"[{event.author}] {part.text.strip()[:600]}")
    return await team_sessions.get_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=session.id
    )


final_session = await run_team(QUESTION)

[procurement_coordinator] -> data_analyst


[data_analyst] -> analyze_supplier_monthly_spend


[data_analyst] -> record_finding


Dropping function calls with no matching function response: ['call_1417476']


[data_analyst] -> transfer_to_agent


...

[knowledge_agent] Contract terms for supplier S102 specify an annual contract value of GBP 138,500.00 with a 5.00% price adjustment cap, requiring buyer approval and an executed amendment for increases above the cap. Procurement policy further dictates that adjustments must be supported by a cost breakdown and market comparison, with changes above the cap requiring documented procurement-owner approval and a signed contract amendment.


[procurement_coordinator] -> research_agent


[research_agent] -> get_commodity_trend
[research_agent] -> get_commodity_trend
[research_agent] -> get_commodity_trend


[research_agent] -> record_finding
[research_agent] -> record_finding
[research_agent] -> record_finding


[research_agent] External commodity price data for aluminum, copper, and zinc were retrieved covering July 2023 through December 2025 to evaluate April 2024 cost drivers. Please note that market indices serve as broad macroeconomic context and are not direct proof of an individual supplier's actual costs.


## Inspect shared state

In [13]:
import json

import pandas as pd

state = final_session.state
print("Evidence")
display(pd.DataFrame(state.get("evidence", []))[["evidence_id", "source_type", "citation_label", "is_synthetic", "excerpt"]])
print("Findings")
display(pd.DataFrame(state.get("findings", []))[["finding_id", "agent", "kind", "confidence", "evidence_ids", "statement", "caveats"]])
print("Critique:", json.dumps(state.get("critique"), indent=2))
print("Decision brief:", json.dumps(state.get("decision_brief"), indent=2))

Evidence


,evidence_id,source_type,citation_label,is_synthetic,excerpt
0,E1,database,Purchase-order spend S102 2024-01-01..2025-12-31,True,"24 months, first 2024-01-01 2400.00, last 2025..."
1,E2,database,Contract record SYN-CON-S102,True,"SYN-CON-S102: 2024-01-01..2026-06-30, annual v..."
2,E3,document,Fictional Meridian Industrial Supply Ltd suppl...,True,> SYNTHETIC PORTFOLIO DATA. This fictional doc...
3,E4,document,Procurement policy: price adjustments,True,> SYNTHETIC PORTFOLIO DATA. This fictional doc...
4,E5,document,Procurement policy: supplier performance and r...,True,> SYNTHETIC PORTFOLIO DATA. This fictional doc...
5,E6,external,world_bank_pink_sheet aluminum monthly price,False,aluminum (($/mt)): 2023-10-01 2192.000000 -> 2...
6,E7,external,world_bank_pink_sheet copper monthly price,False,copper (($/mt)): 2023-10-01 7937.000000 -> 202...
7,E8,external,world_bank_pink_sheet zinc monthly price,False,zinc (($/mt)): 2023-10-01 2449.000000 -> 2024-...
8,E9,external,world_bank_pink_sheet aluminum monthly price,False,aluminum (($/mt)): 2023-07-01 2160.000000 -> 2...
9,E10,external,world_bank_pink_sheet copper monthly price,False,copper (($/mt)): 2023-07-01 8477.000000 -> 202...


Findings


,finding_id,agent,kind,confidence,evidence_ids,statement,caveats
0,F1,data_analyst,fact,high,[E1],"For supplier S102, March 2024 spend was GBP 12...",[Data is synthetic.]
1,F2,knowledge_agent,fact,high,"[E2, E3]",Supplier S102 has an annual contract value of ...,[Data is synthetic and generated for local tes...
2,F3,knowledge_agent,fact,high,"[E3, E4]",Documented annual adjustments for supplier S10...,[Data is synthetic and generated for local tes...
3,F4,research_agent,fact,high,[E6],Aluminum prices increased by 14.32% from Octob...,[Market indices are general context and not di...
4,F5,research_agent,fact,high,[E7],Copper prices increased by 19.24% from October...,[Market indices are general context and not di...
5,F6,research_agent,fact,high,[E8],Zinc prices increased by 11.60% from October 2...,[Market indices are general context and not di...
6,F7,knowledge_agent,fact,high,"[E2, E3]",Supplier S102 has an annual contract value of ...,[Synthetic portfolio data generated for local ...
7,F8,knowledge_agent,fact,high,[E3],Any price adjustment request above the 5% cap ...,[Synthetic portfolio data generated for local ...
8,F9,knowledge_agent,fact,high,[E4],Procurement policy requires that price adjustm...,[Synthetic portfolio data generated for local ...
9,F10,research_agent,fact,high,[E9],"Aluminum prices rose from $2,226/mt in March 2...","[Market indices are context, not proof of a su..."


Critique: {
  "verdict": "approved",
  "issues": [],
  "requested_follow_ups": []
}
Decision brief: {
  "headline": "Supplier S102 experienced a significant spend spike in April 2024 coinciding with commodity price peaks, which must be strictly managed against contractual caps and procurement policies.",
  "verified_facts": [
    "For supplier S102, March 2024 spend was GBP 1204.80 and April 2024 spend was GBP 2313.80, resulting in the largest absolute month-over-month change of GBP 1109.00 (92.05%) in April 2024. [F1]",
    "Supplier S102 has an annual contract value of GBP 138,500.00 with a price adjustment cap of 5.00% under agreement SYN-CON-S102 spanning 2024-01-01 to 2026-06-30. [F2]",
    "Documented annual adjustments for supplier S102 may not exceed 5%, require buyer approval and cost breakdown if above cap, and must follow procurement policy requiring independent market comparison and signed contract amendments. [F3]",
    "Aluminum prices increased by 14.32% from October 202